### Context
split_dataset.py and stastic.py (original files) both operate on 10-second .npz files (from the author's Baidu zip)

Each record contains: 
1. .hea (2 leads, 250 Hz, about 9.2M samples)
2. .dat signal
3. .atr rhythm annotations

There are 23 10 hour double-lead ECGs of interest. 

numpy.savez is used to save a .npz file which is a zipped archive of files named after their variables. 

In [22]:
import wfdb
import numpy as np
import os

In [23]:
record = wfdb.rdrecord('/Users/tristanmartin/Desktop/IIB PROJECT/datasets/MIT-BIH-AFDB/04015')
ann = wfdb.rdann('/Users/tristanmartin/Desktop/IIB PROJECT/datasets/MIT-BIH-AFDB/04015', 'atr')

print(type(record.p_signal))
num_samples, num_leads = np.shape(record.p_signal)
print(f'There are {num_samples} samples with {num_leads} leads each.')
print(f'These are samples at which the {len(ann.aux_note)} annotations occur: {ann.sample}.')
print(f'These are the corresponding labels: {ann.aux_note}.')

<class 'numpy.ndarray'>
There are 9205760 samples with 2 leads each.
These are samples at which the 15 annotations occur: [     30  102584  119604  121773  122194  133348  166857 1096245 1098054
 1135296 1139595 1422436 1423548 1459277 1460416].
These are the corresponding labels: ['(N', '(AFIB', '(N', '(AFIB', '(N', '(AFIB', '(N', '(AFIB', '(N', '(AFIB', '(N', '(AFIB', '(N', '(AFIB', '(N'].


### Target Format
dataset.py's _load_all_segments() loads {record_id}.npz and expects two arrays: 
1. segments (n_windows, 2500, 1): 10 seconds x 2500 Hz with a single lead
2. labels (n_windows,): contains strings 'N' and 'AFIB'

In [24]:
from scipy.signal import butter, sosfiltfilt
import neurokit2

In [25]:
# Some of the following was generated by Claude. It is used to convert the raw data files into 
# 10 second segments for use in this repository. The quality assessment and bandpass 
# filtering functions were written by me. 

RAW_DIR = '/Users/tristanmartin/Desktop/IIB PROJECT/datasets/MIT-BIH-AFDB'
OUT_DIR = '/Users/tristanmartin/Desktop/IIB PROJECT/MGCNet-Research/afdb_dataset/data'

FS = 250                # AFDB is already 250 Hz, so no resampling
WIN_LEN = 10 * FS       # 10 s windows -> 2500 samples
LEAD = 0                # ECG1 only; the model is single-lead (in_channels: 1)
KEEP_LABELS = {'N', 'AFIB'}
QUALIFIED = {'Excellent', 'Barely acceptable'}   # zhao2018 grades counted as "qualified"; 'Unacceptable' is dropped


# Assigns a label to every sample, not just the changes of rhythms. 
# Samples before the first annotation stay ''.

def bandpass_signal(record, low_f = 0.5, high_f = 40, order = 4):
    signal = record.p_signal[:, 0]
    fs = record.fs
    sos = butter(order, [low_f, high_f], btype = 'bandpass', fs = fs, output = 'sos')
    return sosfiltfilt(sos, signal)

def rhythm_per_sample(ann, sig_len):
    per_sample = np.full(sig_len, '', dtype=object)
    for i, (start, note) in enumerate(zip(ann.sample, ann.aux_note)):
        end = ann.sample[i + 1] if i + 1 < len(ann.sample) else sig_len
        per_sample[start:end] = note.strip('(').strip('\x00')
    return per_sample

# Keep a window only if one rhythm covers all of it and that rhythm is N or AFIB.
# Windows that cross a rhythm change, or contain AFL/J/unlabelled samples, are dropped (returns None).
def label_window(window_labels):
    unique = set(window_labels) # Checks the number of different labels in the window
    if len(unique) != 1:
        return None
    label = unique.pop() # Assigns the sole label in unique to 'label'
    return label if label in KEEP_LABELS else None

def convert_record(record_id):
    path = os.path.join(RAW_DIR, record_id)
    sig = bandpass_signal(wfdb.rdrecord(path, channels=[LEAD]))
    per_sample = rhythm_per_sample(wfdb.rdann(path, 'atr'), len(sig))

    segments, labels = [], []
    for start in range(0, len(sig) - WIN_LEN + 1, WIN_LEN):   # non-overlapping windows
        label = label_window(per_sample[start:start + WIN_LEN])
        if label is None:
            continue
        window = sig[start:start + WIN_LEN]
        if not quality_assessment(window):
            continue
        segments.append(window[:, None])                         # (2500, 1)
        labels.append(label)

    np.savez(os.path.join(OUT_DIR, record_id + '.npz'),
             segments=np.stack(segments).astype(np.float32),    # (n_windows, 2500, 1)
             labels=np.array(labels))                            # (n_windows,)
    return labels

# Zhao et al. (2018) signal quality check, as cited in the MGCNet paper, via NeuroKit2.
# Fuses R-peak agreement between two detectors (qSQI), kurtosis (kSQI), QRS-band power (pSQI) and
# baseline wander (basSQI) into 'Excellent' / 'Barely acceptable' / 'Unacceptable'.
# Expects an already bandpassed window. Returns True if the window is "qualified".
def quality_assessment(window):
    try:
        grade = neurokit2.ecg_quality(window, sampling_rate=FS, method='zhao2018', approach='simple')
    except Exception:   # e.g. too few R-peaks found to compute the indices
        return False
    return grade in QUALIFIED

os.makedirs(OUT_DIR, exist_ok=True)
with open(os.path.join(RAW_DIR, 'RECORDS')) as f:
    records = [r.strip() for r in f if r.strip()]

for record_id in records:
    if not os.path.exists(os.path.join(RAW_DIR, record_id + '.dat')):
        print(f'{record_id}: no .dat file, skipped')   # 00735 and 03665 have annotations only
        continue
    labels = convert_record(record_id)
    print(f'{record_id}: N={labels.count("N")}, AFIB={labels.count("AFIB")}')

00735: no .dat file, skipped
03665: no .dat file, skipped
04015: N=3466, AFIB=17
04043: N=2783, AFIB=706
04048: N=3431, AFIB=28
04126: N=3524, AFIB=127
04746: N=1660, AFIB=1940
04908: N=3341, AFIB=301
04936: N=652, AFIB=2613
05091: N=3587, AFIB=4
05121: N=1239, AFIB=2189
05261: N=3618, AFIB=39
06426: N=123, AFIB=2070
06453: N=3144, AFIB=26
06995: N=1942, AFIB=1729
07162: N=0, AFIB=3400
07859: N=0, AFIB=3611
07879: N=1446, AFIB=2214
07910: N=3043, AFIB=582
08215: N=708, AFIB=2966
08219: N=2832, AFIB=750
08378: N=2726, AFIB=751
08405: N=1020, AFIB=2651
08434: N=3535, AFIB=140
08455: N=1130, AFIB=2469
